In [ ]:
from libraries import *
from parameters import *

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.decomposition import PCA
from sklearn.preprocessing import scale 

from sklearn.preprocessing import StandardScaler

import statsmodels.formula.api as smf

from sklearn.neural_network import MLPRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

from sklearn.decomposition import FastICA

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from matplotlib import pyplot

import multiprocessing
import time

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
adata = sc.read_h5ad('./DATA/sc_training.h5ad')


In [ ]:
## Remove KOs that have less than 15 cells
kk = adata.obs.condition.value_counts()[adata.obs.condition.value_counts() < 15]
adata = adata[[x not in kk.index for x in adata.obs.condition],:].copy()

testKOs = list(adata.var_names)

trainKOs = adata.obs.condition.unique()

## Remove KO genes that are not expressed
notavailKOs = [x for x in trainKOs if x not in adata.var_names]
adata = adata[[x not in ['Fzd1', 'P2rx7'] for x in adata.obs.condition],:].copy()

trainKOs = adata.obs.condition.unique()
trainKOs = [x for x in trainKOs if x not in ['Unperturbed']]


allKOs = testKOs

In [ ]:
adata.X = adata.layers['rawcounts'].copy()
adata.obs['n_genes'] = (adata.X != 0).sum(1).A1

mt_gene_mask = adata.var_names.str.startswith('mt-')
assert mt_gene_mask.sum() > 0, 'Wrong mt prefix'
adata.obs['mt_frac'] = adata.X[:, mt_gene_mask].sum(1).A1 / adata.obs['n_umis']


Identify marker genes of the cell states

In [ ]:
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.scale(adata, max_value=10)

sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, metric="euclidean")

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.8)
sc.tl.diffmap(adata)

sc.tl.rank_genes_groups(adata, groupby="state", n_genes=2000, method="t-test_overestim_var")
markerGenes = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
markerGenes = markerGenes.iloc[0:10,:]

markerGenes = list(markerGenes.melt().value)

Generate response matrix

In [ ]:
myDF_2 = pd.crosstab(adata.obs.condition,
                     adata.obs.state)
Y = myDF_2.div(myDF_2.sum(axis=1), axis=0)
Y = Y[['progenitor', 
       'effector',
       'terminal exhausted',
       'cycling',
       'other']]

Cluster the KOs based on the cell state percent distribution

In [ ]:
respAnnDat = sc.AnnData(X=Y)
#sc.pp.scale(koGuidesAnnDat)
#sc.pp.pca(respAnnDat, svd_solver='arpack')
sc.pp.neighbors(respAnnDat)
sc.tl.leiden(respAnnDat, resolution=1.5)
sc.tl.umap(respAnnDat)
sc.pl.umap(respAnnDat, 
           color='leiden',
           size=100,  
           legend_fontoutline=3, 
           legend_loc = 'center',
           legend_fontsize=14,
           legend_fontweight='normal')

In [ ]:
KOClusters = respAnnDat.obs
KOClusters = KOClusters.sort_values(by=['leiden'])

Visualize the covariance of the KOs based on the response variable

In [ ]:
M = pd.DataFrame(np.corrcoef(Y))
M.index = Y.index
M.columns = Y.index
M = M.loc[KOClusters.index, KOClusters.index]
 

fig, ax = plt.subplots(figsize=(20,20))         # Sample figsize in inches
sns.heatmap(M,  linewidths=.5, ax=ax)

Generate the correlation matrix between the KO genes and the marker genes

In [ ]:
expVar = pd.DataFrame(adata.X)
expVar.columns = adata.var_names

geneList = [x for x in list(set(allKOs+markerGenes)) if x in expVar.columns] 
myGenesExp = expVar.loc[:,geneList]
corCoefs = pd.DataFrame(np.corrcoef(np.transpose(myGenesExp)))

corCoefs.columns = geneList
corCoefs.index = geneList

corCoefs.replace(np.nan, 0, inplace=True)
corCoefs = corCoefs.loc[allKOs,markerGenes]

koClustersOrder = [x for x in KOClusters.index if x in corCoefs.index]

Identify the markes genes whose correlation with the KO genes are informative

In [ ]:
forest = RandomForestClassifier(n_estimators=500,
                                random_state=1)
forest.fit(corCoefs.loc[koClustersOrder,], KOClusters.loc[koClustersOrder,].values.ravel())

importance = forest.feature_importances_
sorted_indices = np.argsort(importance)[::-1]

pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()


for i in range(2,len(corCoefs.columns),1):
    print(i)
    myX =  corCoefs.iloc[:,sorted_indices[0:i]]
    
    X_train = myX.loc[trainKOs, ]
    Y = Y.loc[X_train.index,:]
        
    model = RandomForestRegressor(n_estimators=400, max_depth=10, random_state=2)
        
    cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1)
    
    n_scores = cross_val_score(model, X_train, Y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    n_scores = absolute(n_scores)
    
    print('MAE: %.3f (%.3f)' % (np.mean(n_scores), np.std(n_scores)))

Select the top 10 informative marker genes

In [ ]:
markerCoefs_reduced = corCoefs.iloc[:,sorted_indices[0:10]]

Identify the effect of each KO on the genes correcting for possible confounders

In [ ]:
#### Here run "python RunRegression.py" before continuing with the remaining cells

In [ ]:
allRes = pd.read_csv("AllResults_cellStateRegressedOut.csv", index_col=None)

In [ ]:
allRes

In [ ]:
coefs = pd.pivot(allRes[['guides','coef', 'respGene']],
                 index='guides', columns='respGene', 
                 values='coef')
coefs = coefs.loc[coefs.index.isin(coefs.columns),]
coefs = coefs.astype(float)


pvals = pd.pivot(allRes[['guides','pval', 'respGene']],
                 index='guides', columns='respGene', 
                 values='pval')
pvals = pvals.loc[pvals.index.isin(pvals.columns),]
pvals = pvals.astype(float)


In [ ]:
coefs.iloc[pvals > 0.1] = 0

In [ ]:
coefs = coefs.T
pvals = pvals.T

In [ ]:
pca = PCA(n_components=25)
pca.fit(coefs)
print(pca.explained_variance_ratio_)
print(sum(pca.explained_variance_ratio_))

coefs_reduced = pd.DataFrame(pca.fit_transform(coefs))
coefs_reduced.index = coefs.index
coefs_reduced.columns = ["pcaCoef" + str(x) for x in range(0,len(coefs_reduced.columns),1)] 

In [ ]:
forest = RandomForestClassifier(n_estimators=500,
                                random_state=1)
forest.fit(coefs_reduced.loc[koClustersOrder,], KOClusters.loc[koClustersOrder,].values.ravel())

importance = forest.feature_importances_
sorted_indices = np.argsort(importance)[::-1]

pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

for i in range(2,25,1):
    print(i)
    coefs_reduced_tmp = coefs_reduced.iloc[:,sorted_indices[0:i]]

    myX =  pd.concat([coefs_reduced_tmp.loc[markerCoefs_reduced.index,:], markerCoefs_reduced], axis=1)
    myX_scaled = pd.DataFrame(StandardScaler(with_std=False).fit(myX).transform(myX.astype(float)))
    myX_scaled.index = myX.index
    myX_scaled.columns = myX.columns
    
    
    X_train = myX_scaled.loc[trainKOs, ]
    Y = Y.loc[X_train.index,:]
    
    model = RandomForestRegressor(n_estimators=400, max_depth=10, random_state=2)
    
    cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1)
    
    n_scores = cross_val_score(model, X_train, Y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    n_scores = absolute(n_scores)
    
    print(n_scores)
    
    print('MAE: %.3f (%.3f)' % (np.mean(n_scores), np.std(n_scores)))

In [ ]:
coefs_reduced = coefs_reduced.iloc[:,sorted_indices[0:9]]

In [ ]:
coefs_reduced

In [ ]:
myX =  pd.concat([coefs_reduced.loc[markerCoefs_reduced.index,:], markerCoefs_reduced], axis=1)
myX_scaled = pd.DataFrame(StandardScaler(with_std=False).fit(myX).transform(myX.astype(float)))
myX_scaled.index = myX.index
myX_scaled.columns = myX.columns
 
X_train = myX_scaled.loc[trainKOs, ]
Y = Y.loc[X_train.index,:]

model = RandomForestRegressor(n_estimators=400, max_depth=10, random_state=2)
model.fit(X_train,Y)



In [ ]:
X_val = myX_scaled
valPreds = pd.DataFrame(model.predict(X_val))
valPreds.index=myX_scaled.index

In [ ]:
valPreds.columns = ['progenitor', 'effector','terminal exhausted','cycling','other']

In [ ]:
valPreds = valPreds.sort_values(by=['progenitor'], ascending=False)

In [ ]:
valPreds["cycling_constraint"] = [1 for x in valPreds.cycling if x > 0.05]

In [ ]:
myRes = valPreds.loc[:,["progenitor", "cycling_constraint"]]
myRes.columns = ["objective", "cycling_constraint"]
myRes.to_csv("part_a_output.csv")

In [ ]:
valPreds["objective"] = (valPreds["progenitor"] / 0.0675) + (valPreds["effector"] / 0.2097) - (valPreds["terminal exhausted"] / 0.3134) + (valPreds["cycling"] / 0.3921) 

In [ ]:
partB = valPreds.loc[:,["objective", "cycling_constraint"]]

In [ ]:
partB = partB.sort_values(by=['objective'], ascending=False)

In [ ]:
partB.to_csv("part_b_output.csv")